# Training Random Forest + Ekspor TFLite — URL Classifier
**Google Colab | Dataset 200.000 Balanced | Export ke Android**

---
Notebook ini melakukan training final dan mengekspor model ke format TFLite
yang kompatibel dengan aplikasi Android.

**Alur:**
1. Load dataset → ambil 200.000 sampel balanced
2. Verifikasi fungsi ekstraksi fitur (harus identik dengan Android)
3. Training model Random Forest
4. Evaluasi performa (accuracy, F1, confusion matrix)
5. Ekspor ke TFLite via pipeline: **sklearn → ONNX → TFLite**
6. Verifikasi model TFLite dengan beberapa contoh URL

**Perbedaan utama dari training sebelumnya:**
- Menggunakan `zipmap=False` saat konversi ONNX → **menghindari error GATHER di Android**
- Fungsi ekstraksi fitur **identik** dengan kode Android (`RandomForestClassifier.kt`)

## Cell 1 — Install Library

Instalasi `skl2onnx` dan `onnx2tf` untuk pipeline konversi ke TFLite.

In [ ]:
# Library standar ML
!pip install -q --upgrade scikit-learn seaborn

# Pipeline konversi: sklearn → ONNX
# skl2onnx: konverter resmi sklearn ke ONNX
!pip install -q skl2onnx onnx onnxruntime

# Pipeline konversi: ONNX → TFLite
# onnx2tf: library aktif yang menangani banyak ONNX op termasuk RF
!pip install -q onnx2tf

print('Semua library berhasil diinstall.')

## Cell 2 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import time
import os
import re
import subprocess
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

import tensorflow as tf
import onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

print(f'scikit-learn : {__import__("sklearn").__version__}')
print(f'TensorFlow   : {tf.__version__}')
print(f'ONNX         : {onnx.__version__}')

## Cell 3 — Mount Google Drive & Konfigurasi Path

Sesuaikan `DATASET_PATH` dan `SAVE_PATH` dengan struktur folder Google Drive kamu.

**Format dataset** — CSV dengan kolom minimal:
```
label, domain_length, digit_count, dot_count, delimiter_count,
suspicious_word_count, digit_letter_ratio, max_sequential_digits
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
# SESUAIKAN PATH INI
# ================================================================
DATASET_PATH = '/content/drive/MyDrive/Tugas Akhir/Dataset/dataset_100rb.csv'
SAVE_PATH    = '/content/drive/MyDrive/Tugas Akhir/Training/RF/'
# ================================================================

os.makedirs(SAVE_PATH, exist_ok=True)

FEATURE_COLS = [
    'domain_length',
    'digit_count',
    'dot_count',
    'delimiter_count',
    'suspicious_word_count',   # F5: COUNT, bukan boolean
    'digit_letter_ratio',
    'max_sequential_digits'    # F7: panjang MAX, bukan boolean
]
LABEL_COL = 'label'

df = pd.read_csv(DATASET_PATH)
print(f'Dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'Distribusi label:\n{df[LABEL_COL].value_counts()}')
df.head()

## Cell 4 — Fungsi Ekstraksi Fitur

> **PENTING**: Fungsi ini HARUS IDENTIK dengan `extractLexicalFeatures()` di `RandomForestClassifier.kt`.
> Jika ada perbedaan sekecil apapun, model akan salah mengklasifikasi di Android.

Digunakan untuk:
1. Verifikasi bahwa fitur di dataset konsisten dengan cara hitung Android
2. Mengekstrak fitur dari URL baru jika dataset hanya berisi URL+label

**Perbedaan dengan training sebelumnya:**
- F5 `suspicious_word_count`: sekarang **COUNT** (bisa 0,1,2,...), bukan boolean
- F7 `max_sequential_digits`: sekarang **panjang maksimal** digit berurutan, bukan boolean

In [ ]:
# Daftar kata mencurigakan — HARUS SAMA dengan SUSPICIOUS_WORDS di Android
SUSPICIOUS_WORDS = {
    'porn', 'sex', 'xxx', 'adult', 'cam', 'tube', 'bokep', 'hentai',
    'nude', 'gay', 'lesbian', 'erotic', 'mature', 'amateur',
    'creampie', 'milf', 'bbw', 'naked', 'porno', 'anal',
    'pussy', 'cock', 'cumshot', 'orgasm', 'xvideos', 'xnxx',
    'xhamster', 'redtube', 'youporn', 'brazzers', 'onlyfans'
}

def extract_features(domain: str) -> list:
    """
    Ekstrak 7 fitur leksikal dari domain URL.
    Input : 'youporn.com' (domain setelah normalisasi, dengan TLD)
    Output: [F1, F2, F3, F4, F5, F6, F7]
    """
    lower = domain.lower()

    digits  = sum(c.isdigit() for c in lower)           # F2
    letters = sum(c.isalpha() for c in lower)           # untuk F6

    # F5: COUNT kata mencurigakan (identik dgn SUSPICIOUS_WORDS.count{} di Android)
    suspicious_word_count = sum(1 for w in SUSPICIOUS_WORDS if w in lower)

    # F6: digit / letter ratio
    digit_letter_ratio = digits / letters if letters > 0 else float(digits)

    # F7: panjang maksimal urutan digit (identik dgn Regex("\\d+").findAll().maxOfOrNull di Android)
    sequences = re.findall(r'\d+', lower)
    max_sequential_digits = max((len(s) for s in sequences), default=0)

    return [
        len(domain),                                                    # F1: domain_length
        digits,                                                         # F2: digit_count
        lower.count('.'),                                               # F3: dot_count
        sum(1 for c in lower if not c.isalnum() and c != '.'),         # F4: delimiter_count
        suspicious_word_count,                                          # F5: suspicious_word_count
        digit_letter_ratio,                                             # F6: digit_letter_ratio
        max_sequential_digits,                                          # F7: max_sequential_digits
    ]

# Verifikasi dengan contoh konkret — cocokkan dengan logcat Android
test_cases = [
    ('youporn.com',       1, 'F5=1(porn), F7=0'),
    ('xvideos.com',       1, 'F5=1(xvideos), F7=0'),
    ('xxx18hub.com',      1, 'F5=1(xxx), F7=2(dari 18)'),
    ('bba021.com',        0, 'F5=0, F7=3(dari 021)'),
    ('google.com',        0, 'semua fitur rendah'),
    ('hand-job-porn.com', 1, 'F5=1(porn), F4=1(dash), F7=0'),
]

print(f'{"Domain":<25} {"Lbl":<5} {"F1":>4} {"F2":>4} {"F3":>4} {"F4":>4} {"F5":>4} {"F6":>7} {"F7":>4}   Catatan')
print('-' * 95)
for domain, label, note in test_cases:
    f = extract_features(domain)
    print(f'{domain:<25} {label:<5} {f[0]:>4} {f[1]:>4} {f[2]:>4} {f[3]:>4} {f[4]:>4} {f[5]:>7.3f} {f[6]:>4}   {note}')

## Cell 5 — Sampling 200.000 Data & Validasi Konsistensi Fitur

Jika dataset sudah memiliki kolom fitur yang dihitung dengan cara yang sama,
kita bisa menggunakannya langsung.

Cell ini juga memverifikasi bahwa nilai F5 dan F7 di dataset konsisten
dengan definisi yang dipakai Android.

In [ ]:
N_PER_CLASS = 100_000

df_safe = df[df[LABEL_COL] == 0].sample(n=N_PER_CLASS, random_state=42)
df_porn = df[df[LABEL_COL] == 1].sample(n=N_PER_CLASS, random_state=42)
df_bal  = pd.concat([df_safe, df_porn]).sample(frac=1, random_state=42).reset_index(drop=True)
df_bal  = df_bal.drop_duplicates(subset=FEATURE_COLS + [LABEL_COL])
df_bal  = df_bal.dropna(subset=FEATURE_COLS + [LABEL_COL]).reset_index(drop=True)

print(f'Dataset setelah sampling: {len(df_bal):,} baris')
print(f'Distribusi: {df_bal[LABEL_COL].value_counts().to_dict()}')
print()

# Verifikasi nilai F5 dan F7 di dataset
# F5 seharusnya integer (0,1,2,...), bukan hanya 0 dan 1
# F7 seharusnya integer (0,1,2,...), bukan hanya 0 dan 1
print('Distribusi nilai F5 (suspicious_word_count):')
print(df_bal['suspicious_word_count'].value_counts().sort_index().head(10))
print()
print('Distribusi nilai F7 (max_sequential_digits):')
print(df_bal['max_sequential_digits'].value_counts().sort_index().head(10))

# Jika F5 dan F7 hanya berisi 0 dan 1 (boolean), dataset perlu diperbarui
f5_unique = sorted(df_bal['suspicious_word_count'].unique())
f7_unique = sorted(df_bal['max_sequential_digits'].unique())

if set(f5_unique) <= {0, 1}:
    print('\n⚠️  F5 hanya berisi {0,1} — kemungkinan boolean dari dataset lama.')
    print('   Idealnya rekalkulasi fitur dari URL asli menggunakan fungsi extract_features().')
else:
    print('\n✅ F5 menggunakan integer count — sesuai dengan Android.')

if set(f7_unique) <= {0, 1}:
    print('⚠️  F7 hanya berisi {0,1} — kemungkinan boolean dari dataset lama.')
else:
    print('✅ F7 menggunakan integer max length — sesuai dengan Android.')

## Cell 6 — (Opsional) Hitung Ulang Fitur dari URL

Jalankan cell ini **HANYA JIKA** Cell 5 mendeteksi F5/F7 masih boolean,
atau jika dataset hanya memiliki kolom `url` dan `label`.

Proses ini memakan waktu karena menghitung fitur satu per satu untuk setiap URL.

In [ ]:
# Ubah RECOMPUTE_FEATURES = True jika perlu menghitung ulang fitur
RECOMPUTE_FEATURES = False

if RECOMPUTE_FEATURES:
    if 'url' not in df_bal.columns:
        raise ValueError('Kolom url tidak ditemukan di dataset. Tidak bisa menghitung ulang fitur.')

    print('Menghitung ulang fitur dari URL... (bisa memakan waktu beberapa menit)')
    t0 = time.time()

    feat_list = df_bal['url'].apply(extract_features).tolist()
    feat_df   = pd.DataFrame(feat_list, columns=FEATURE_COLS)

    # Update kolom fitur di dataset
    for col in FEATURE_COLS:
        df_bal[col] = feat_df[col].values

    print(f'Selesai dalam {time.time()-t0:.1f} detik')
    print('\nDistribusi F5 setelah rekalkulasi:')
    print(df_bal['suspicious_word_count'].value_counts().sort_index().head(10))
    print('\nDistribusi F7 setelah rekalkulasi:')
    print(df_bal['max_sequential_digits'].value_counts().sort_index().head(10))
else:
    print('Melewati rekalkulasi fitur (RECOMPUTE_FEATURES = False)')

## Cell 7 — Persiapan Data Training

**Tidak ada scaling** — Random Forest tidak sensitif terhadap skala fitur
karena menggunakan split berbasis threshold, bukan jarak.

In [ ]:
X = df_bal[FEATURE_COLS].values.astype(np.float32)
y = df_bal[LABEL_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # Jaga proporsi kelas di train & test
)

# Tidak perlu scaling untuk Random Forest
X_train_scaled = X_train
X_test_scaled  = X_test

print(f'Training set : {len(X_train):,} baris')
print(f'Testing set  : {len(X_test):,} baris')
print(f'\nBentuk input : {X_train.shape}  → {X_train.shape[1]} fitur')
print(f'Distribusi train: {np.bincount(y_train)}')
print(f'Distribusi test : {np.bincount(y_test)}')

## Cell 8 — Input Hyperparameter Terbaik

Notebook akan mencoba membaca `best_params.json` dari hasil tuning.
Jika tidak ada, gunakan nilai default dari `rf_training.ipynb` sebelumnya.

In [ ]:
# Coba load dari hasil tuning otomatis
json_path = SAVE_PATH + 'best_params.json'

try:
    with open(json_path) as f:
        best_params = json.load(f)
    print(f'✅ Loaded dari: {json_path}')
except FileNotFoundError:
    # Fallback: nilai default jika tuning belum dijalankan
    # (sesuai Tabel III.2 — nilai tengah dari rentang yang diuji)
    best_params = {
        'n_estimators': 200,
        'max_depth'   : 10,
        'max_features': 'sqrt',
    }
    print('⚠️  best_params.json tidak ditemukan. Menggunakan default:')

print('\nHyperparameter yang akan digunakan:')
for k, v in best_params.items():
    print(f'  {k:<22}: {v}')

## Cell 9 — Training Model

In [ ]:
model = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1      # Gunakan semua CPU yang tersedia
)

print('Memulai training...')
t0 = time.time()
model.fit(X_train_scaled, y_train)
durasi = time.time() - t0

print(f'Training selesai dalam {durasi:.1f} detik ({durasi/60:.2f} menit)')
print(f'Jumlah trees   : {len(model.estimators_)}')
print(f'Max depth      : {model.max_depth}')

## Cell 10 — Evaluasi Model

In [ ]:
y_pred = model.predict(X_test_scaled)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)

print('=' * 45)
print('  Evaluasi pada Test Set (20%)')
print('=' * 45)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('=' * 45)

report_str = classification_report(
    y_test, y_pred,
    target_names=['Aman (0)', 'Pornografi (1)'],
    digits=4
)
print('\nClassification Report:')
print(report_str)

# Simpan classification report TXT
with open('/content/classification_report_rf.txt', 'w') as f:
    f.write('Classification Report — Random Forest URL Classifier\n')
    f.write('=' * 50 + '\n\n')
    f.write(report_str)
    f.write(f'\nAccuracy   : {acc:.4f}\n')
    f.write(f'Precision  : {prec:.4f}\n')
    f.write(f'Recall     : {rec:.4f}\n')
    f.write(f'F1-Score   : {f1:.4f}\n')

# Simpan classification report CSV
pd.DataFrame(
    classification_report(y_test, y_pred,
        target_names=['Aman (0)', 'Pornografi (1)'],
        output_dict=True)
).T.to_csv('/content/classification_report_rf.csv')

print('✅ classification_report_rf.txt + .csv tersimpan')

# Bar chart evaluasi
fig, ax = plt.subplots(figsize=(9, 5))
metrics_vals  = [acc, prec, rec, f1]
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
bars = ax.bar(metrics_names, metrics_vals, color=colors, alpha=0.85, edgecolor='white')
for b, v in zip(bars, metrics_vals):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
            f'{v*100:.2f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title('Metrik Evaluasi — Random Forest URL Classifier', fontweight='bold', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/metrics_barchart_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ metrics_barchart_rf.png tersimpan')

## Cell 11 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'True Negative  (TN — URL aman, diprediksi aman)    : {tn:,}')
print(f'False Positive (FP — URL aman, salah blokir)       : {fp:,}')
print(f'False Negative (FN — URL porno, tidak terdeteksi)  : {fn:,}')
print(f'True Positive  (TP — URL porno, berhasil terdeteksi): {tp:,}')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Aman (0)', 'Porno (1)'],
    yticklabels=['Aman (0)', 'Porno (1)'],
    linewidths=0.5, annot_kws={'size': 13, 'weight': 'bold'}
)
ax.set_title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Prediksi', fontsize=11)
ax.set_ylabel('Label Sebenarnya', fontsize=11)
plt.tight_layout()
plt.savefig(SAVE_PATH + 'confusion_matrix_rf.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 12 — Feature Importance

In [ ]:
imp_df = pd.DataFrame({
    'Fitur'     : FEATURE_COLS,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(imp_df['Fitur'], imp_df['Importance'], color='#1E88E5', edgecolor='white')
for bar, val in zip(bars, imp_df['Importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_PATH + 'feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()

print(imp_df.sort_values('Importance', ascending=False).to_string(index=False))

## Cell 13 — Cek Overfitting & Simpan Joblib

In [ ]:
# Bandingkan performa di train vs test
y_pred_train = model.predict(X_train_scaled)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test  = accuracy_score(y_test, y_pred)
f1_train  = f1_score(y_train, y_pred_train)
f1_test   = f1_score(y_test, y_pred)

print('Perbandingan performa (cek overfitting):')
print(f'  Training  — Accuracy: {acc_train:.4f} | F1: {f1_train:.4f}')
print(f'  Testing   — Accuracy: {acc_test:.4f} | F1: {f1_test:.4f}')
print(f'  Gap Acc   : {acc_train - acc_test:.4f}', end=' ')
if acc_train - acc_test < 0.02:
    print('✅ Tidak overfitting')
elif acc_train - acc_test < 0.05:
    print('⚠️ Sedikit overfitting')
else:
    print('❌ Overfitting — pertimbangkan kurangi max_depth')

# Simpan model sklearn
joblib_path = SAVE_PATH + 'rf_model.joblib'
joblib.dump(model, joblib_path, compress=3)
print(f'\n✅ Model tersimpan: {joblib_path}')

# Simpan model_info_rf.txt
fi_sorted = sorted(zip(FEATURE_COLS, model.feature_importances_), key=lambda x: x[1], reverse=True)
model_info_lines = [
    'Model Info — Random Forest URL Classifier',
    '=' * 50,
    '',
    f'Jenis Model      : RandomForestClassifier',
    f'n_estimators     : {model.n_estimators}',
    f'max_depth        : {model.max_depth}',
    f'max_features     : {model.max_features}',
    f'random_state     : {model.random_state}',
    f'n_jobs           : {model.n_jobs}',
    '',
    f'Total pohon      : {len(model.estimators_)}',
    f'Jumlah fitur     : {model.n_features_in_}',
    f'Kelas            : {list(model.classes_)}',
    '',
    'Fitur yang digunakan:',
]
for i, col in enumerate(FEATURE_COLS):
    model_info_lines.append(f'  F{i+1}: {col}')

model_info_lines += ['', 'Feature Importance (diurutkan tertinggi):']
for col, imp in fi_sorted:
    bar = '█' * int(imp * 40)
    model_info_lines.append(f'  {col:<30}: {imp:.4f}  {bar}')

model_info_lines += [
    '',
    'Evaluasi pada Test Set:',
    f'  Accuracy  : {acc:.4f}',
    f'  Precision : {prec:.4f}',
    f'  Recall    : {rec:.4f}',
    f'  F1-Score  : {f1:.4f}',
]

with open('/content/model_info_rf.txt', 'w') as f:
    f.write('\n'.join(model_info_lines))
print('✅ model_info_rf.txt tersimpan')

## Cell 14 — Konversi ke TFLite

**Pipeline**: sklearn → ONNX → TFLite

### Kenapa ONNX?
Tidak ada konverter langsung sklearn → TFLite yang andal.
ONNX adalah format perantara yang didukung luas.

### Kenapa `zipmap=False`? ← INI KUNCI MENGHINDARI GATHER ERROR
Secara default, `skl2onnx` menambahkan operator `ZipMap` pada output probabilitas.
Operator ini diimplementasikan sebagai `Gather` di TFLite, yang menyebabkan error:
```
gather index out of bounds. Node number 13 (GATHER) failed to invoke.
```
Dengan `zipmap=False`, output langsung berupa array float `[N, 2]` tanpa `Gather`.

### Format output TFLite:
- Input : `float32 [1, 7]` → 7 nilai fitur
- Output: `float32 [1, 2]` → `[P(aman), P(porno)]`

Android membaca: `outputBuffer.float` (skip P_aman) → `outputBuffer.float` (P_porno)

In [ ]:
print('=== STEP 1: sklearn → ONNX ===')

# zipmap=False: output langsung float array, bukan dictionary
# → MENGHINDARI operator ZipMap yang menyebabkan GATHER error di TFLite
options = {id(model): {'zipmap': False}}

onnx_model = convert_sklearn(
    model,
    initial_types=[('float_input', FloatTensorType([None, 7]))],
    options=options,
    target_opset=11   # Opset 11 kompatibel luas dengan berbagai converter
)

# Tampilkan semua output ONNX
print('Output node di ONNX:')
for i, out in enumerate(onnx_model.graph.output):
    print(f'  [{i}] name="{out.name}"')

# skl2onnx menghasilkan 2 output: [label (int64), probabilities (float32)]
# Kita hanya butuh probabilitas untuk Android
print()
print('=== STEP 2: Hapus output label, simpan hanya probabilities ===')

# Cari output probabilitas
prob_output = None
for out in onnx_model.graph.output:
    if 'prob' in out.name.lower():
        prob_output = out
        break

if prob_output is None:
    # Fallback: ambil output index 1 (biasanya probabilities)
    if len(onnx_model.graph.output) >= 2:
        prob_output = onnx_model.graph.output[1]
    else:
        prob_output = onnx_model.graph.output[0]
    print(f'  Fallback ke output index: {onnx_model.graph.output.index(prob_output)}')

# Ganti output ONNX menjadi hanya probabilities
del onnx_model.graph.output[:]
onnx_model.graph.output.append(prob_output)

print(f'  Output ONNX final: "{prob_output.name}" (shape: [N, 2])')

# Simpan ONNX
onnx_path = '/content/rf_probs.onnx'
onnx.save(onnx_model, onnx_path)
print(f'  ONNX tersimpan: {onnx_path}')

## Cell 15 — Verifikasi ONNX & Konversi ke TFLite

In [ ]:
import onnxruntime as ort

print('=== STEP 3: Verifikasi ONNX dengan onnxruntime ===')

sess = ort.InferenceSession(onnx_path)
inp_name = sess.get_inputs()[0].name
out_name = sess.get_outputs()[0].name
print(f'  Input  : name={inp_name}, shape={sess.get_inputs()[0].shape}')
print(f'  Output : name={out_name}, shape={sess.get_outputs()[0].shape}')

# Test dengan contoh
# youporn.com: F1=10, F2=0, F3=1, F4=0, F5=1(porn), F6=0.0, F7=0
test_vec = np.array([[10, 0, 1, 0, 1, 0.0, 0]], dtype=np.float32)
onnx_out = sess.run([out_name], {inp_name: test_vec})
p_safe, p_adult = onnx_out[0][0][0], onnx_out[0][0][1]
print(f'\n  Test youporn.com: P(aman)={p_safe:.4f}, P(porno)={p_adult:.4f}', end=' ')
print('✅' if p_adult > 0.5 else '❌ (model belum cukup akurat)')

# Juga bandingkan dengan prediksi sklearn langsung
sklearn_prob = model.predict_proba(test_vec)[0]
print(f'  Sklearn output   : P(aman)={sklearn_prob[0]:.4f}, P(porno)={sklearn_prob[1]:.4f}')
print(f'  Selisih (max)    : {max(abs(onnx_out[0][0] - sklearn_prob)):.6f}')

print()
print('=== STEP 4: ONNX → TFLite via onnx2tf ===')

tf_model_dir = '/content/rf_tf_model'
os.makedirs(tf_model_dir, exist_ok=True)

# Konversi ONNX ke TF SavedModel
# -osd: output SavedModel dengan signature
# -nuo: non-verbose output (hapus jika perlu debug)
result = subprocess.run(
    ['onnx2tf', '-i', onnx_path, '-o', tf_model_dir, '-osd', '-nuo'],
    capture_output=True, text=True
)

if result.returncode == 0:
    print('  onnx2tf berhasil!')
else:
    print('  onnx2tf error. Pesan:')
    print(result.stderr[-1000:])
    raise RuntimeError('onnx2tf gagal — coba restart runtime dan install ulang.')

print()
print('=== STEP 5: TF SavedModel → TFLite ===')

converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Quantisasi default (float16)
tflite_model = converter.convert()

tflite_path_local = '/content/url_classifier_rf_v2.tflite'
with open(tflite_path_local, 'wb') as f:
    f.write(tflite_model)

size_kb = len(tflite_model) / 1024
print(f'  TFLite tersimpan: {tflite_path_local}')
print(f'  Ukuran          : {size_kb:.1f} KB')

## Cell 16 — Verifikasi TFLite Model

Pastikan model TFLite:
1. Menerima input `float32 [1, 7]`
2. Menghasilkan output `float32 [1, 2]` — `[P(aman), P(porno)]`
3. Prediksinya konsisten dengan model sklearn asli
4. Bisa mendeteksi URL pornografi dengan benar

In [ ]:
# Load TFLite interpreter
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()

inp_detail  = interp.get_input_details()[0]
outp_detail = interp.get_output_details()[0]

print('=== Info TFLite Model ===')
print(f'Input  : shape={inp_detail["shape"]}, dtype={inp_detail["dtype"].__name__}')
print(f'Output : shape={outp_detail["shape"]}, dtype={outp_detail["dtype"].__name__}')

assert inp_detail['dtype'] == np.float32, 'Input harus float32!'
assert outp_detail['shape'][-1] == 2, 'Output harus [1,2] (P_aman, P_porno)!'
print('\n✅ Format input/output sesuai dengan Android!')

def predict_tflite(domain: str) -> tuple:
    """Prediksi satu domain menggunakan TFLite model."""
    features = np.array([extract_features(domain)], dtype=np.float32)
    interp.set_tensor(inp_detail['index'], features)
    interp.invoke()
    probs = interp.get_tensor(outp_detail['index'])[0]
    return float(probs[0]), float(probs[1])  # P(aman), P(porno)

# Test dengan berbagai URL
test_urls = [
    ('youporn.com',        1),
    ('xvideos.com',        1),
    ('pornhub.com',        1),
    ('xnxx.com',           1),
    ('bokep69.com',        1),
    ('google.com',         0),
    ('youtube.com',        0),
    ('tokopedia.com',      0),
    ('github.com',         0),
    ('kompas.com',         0),
]

print(f'\n{"Domain":<22} {"Label":<7} {"P(aman)":<10} {"P(porno)":<10} {"Prediksi":<10} {"Benar?"}')
print('-' * 72)

benar = 0
for domain, true_label in test_urls:
    p_safe, p_adult = predict_tflite(domain)
    pred = 1 if p_adult > 0.5 else 0
    ok = '✅' if pred == true_label else '❌'
    if pred == true_label:
        benar += 1
    print(f'{domain:<22} {true_label:<7} {p_safe:.4f}     {p_adult:.4f}     {pred:<10} {ok}')

print(f'\nAkurasi contoh: {benar}/{len(test_urls)} ({benar/len(test_urls)*100:.0f}%)')

# Konsistensi TFLite vs sklearn (sampel)
print(f'\n=== Konsistensi TFLite vs sklearn ===')
max_diff = 0
for domain, _ in test_urls:
    feat = np.array([extract_features(domain)], dtype=np.float32)
    sk_prob = model.predict_proba(feat)[0]
    tfl_p_safe, tfl_p_adult = predict_tflite(domain)
    diff = abs(sk_prob[1] - tfl_p_adult)
    max_diff = max(max_diff, diff)
print(f'Selisih probabilitas maksimum: {max_diff:.6f}')
if max_diff < 0.01:
    print('✅ TFLite konsisten dengan sklearn (selisih < 1%)')
else:
    print('⚠️ Ada selisih cukup besar — periksa proses konversi')

# ── Validasi TFLite pada Seluruh Test Set ──
print('\n=== Validasi TFLite pada Seluruh Test Set ===')
tflite_preds_all = []
n_test = len(X_test_scaled)

for i in range(n_test):
    if (i + 1) % 10000 == 0 or i == n_test - 1:
        print(f'  Progress: {i+1}/{n_test}', end='\r')
    feat = X_test_scaled[i:i+1].astype(np.float32)
    interp.set_tensor(inp_detail['index'], feat)
    interp.invoke()
    probs = interp.get_tensor(outp_detail['index'])[0]
    tflite_preds_all.append(float(probs[1]))  # P(porno)

tflite_binary = (np.array(tflite_preds_all) >= 0.5).astype(int)
tflite_acc    = accuracy_score(y_test, tflite_binary)
tflite_f1     = f1_score(y_test, tflite_binary, zero_division=0)
tflite_prec   = precision_score(y_test, tflite_binary, zero_division=0)
tflite_rec    = recall_score(y_test, tflite_binary, zero_division=0)

print(f'\nEvaluasi Sklearn  (Test Set) — Accuracy: {acc*100:.4f}%  F1: {f1:.4f}')
print(f'Evaluasi TFLite   (Test Set) — Accuracy: {tflite_acc*100:.4f}%  F1: {tflite_f1:.4f}')
print(f'Selisih Accuracy                        : {abs(acc - tflite_acc)*100:.4f}%')
print('✅ TFLite konsisten' if abs(acc - tflite_acc) < 0.005 else '⚠️  Selisih > 0.5% — cek konversi')

## Cell 17 — Simpan Semua Output ke Google Drive

File yang tersimpan:
- `rf_model.joblib` — model sklearn (untuk analisis lanjut atau konversi ulang)
- `url_classifier_rf_v2.tflite` — model TFLite untuk Android
- `confusion_matrix_rf.png` — grafik confusion matrix
- `feature_importance_rf.png` — grafik feature importance

**Cara pasang ke Android:**
Salin `url_classifier_rf_v2.tflite` ke folder `app/src/main/assets/` di project Android,
lalu ubah nama menjadi `url_classifier_rf.tflite` (atau sesuaikan di `ClassifierManager.kt`).

In [ ]:
import shutil

# ── Simpan semua file ke Google Drive ──
tflite_drive_path = SAVE_PATH + 'url_classifier_rf_v2.tflite'
onnx_drive_path   = SAVE_PATH + 'rf_model.onnx'

files_to_save = [
    (tflite_path_local,                    tflite_drive_path),
    (onnx_path,                            onnx_drive_path),
    ('/content/classification_report_rf.txt', SAVE_PATH + 'classification_report_rf.txt'),
    ('/content/classification_report_rf.csv', SAVE_PATH + 'classification_report_rf.csv'),
    ('/content/metrics_barchart_rf.png',      SAVE_PATH + 'metrics_barchart_rf.png'),
    ('/content/model_info_rf.txt',            SAVE_PATH + 'model_info_rf.txt'),
]

print('Menyimpan file ke Google Drive...')
for src, dst in files_to_save:
    if os.path.exists(src):
        shutil.copy(src, dst)
        size = os.path.getsize(dst) / 1024
        print(f'  ✅ {os.path.basename(dst):<40} ({size:.1f} KB)')
    else:
        print(f'  ⚠️  {os.path.basename(src)} tidak ditemukan, dilewati')

# ── Simpan evaluation_metrics_rf.json (lengkap) ──
eval_metrics_rf = {
    'model_name'       : 'RandomForest_URL_Classifier',
    'model_type'       : 'RandomForestClassifier',
    'hyperparameters'  : {
        'n_estimators' : model.n_estimators,
        'max_depth'    : model.max_depth,
        'max_features' : model.max_features,
        'random_state' : model.random_state,
    },
    'total_trees'      : len(model.estimators_),
    'n_features'       : model.n_features_in_,
    'feature_names'    : FEATURE_COLS,
    'feature_importance': {col: round(float(imp), 4)
                           for col, imp in zip(FEATURE_COLS, model.feature_importances_)},
    'dataset'          : {
        'total_samples' : len(X),
        'train_samples' : len(X_train),
        'test_samples'  : len(X_test),
        'n_per_class'   : N_PER_CLASS,
        'split_ratio'   : '80/20',
    },
    'evaluation_sklearn': {
        'accuracy' : round(acc,  4),
        'precision': round(prec, 4),
        'recall'   : round(rec,  4),
        'f1_score' : round(f1,   4),
    },
    'evaluation_tflite' : {
        'accuracy'  : round(tflite_acc,  4),
        'precision' : round(tflite_prec, 4),
        'recall'    : round(tflite_rec,  4),
        'f1_score'  : round(tflite_f1,   4),
        'delta_acc' : round(abs(acc - tflite_acc), 4),
    },
    'tflite'            : {
        'size_kb'      : round(len(tflite_model) / 1024, 1),
        'pipeline'     : 'sklearn → ONNX → TFLite',
        'zipmap'       : False,
        'input_dtype'  : inp_detail['dtype'].__name__,
        'input_shape'  : list(map(int, inp_detail['shape'])),
        'output_dtype' : outp_detail['dtype'].__name__,
        'output_shape' : list(map(int, outp_detail['shape'])),
    },
}

json_path = SAVE_PATH + 'evaluation_metrics_rf.json'
with open(json_path, 'w') as f:
    json.dump(eval_metrics_rf, f, indent=2)
print(f'  ✅ {"evaluation_metrics_rf.json":<40} (JSON)')

# ── Ringkasan ──
print(f'\n{"=" * 62}')
print(f'  RINGKASAN PELATIHAN RANDOM FOREST')
print(f'{"=" * 62}')
print(f'  Dataset        : {len(df_bal):,} sampel (100K aman + 100K porno)')
print(f'  Split          : {len(X_train):,} train / {len(X_test):,} test (80/20)')
print(f'  n_estimators   : {model.n_estimators}  |  max_depth: {model.max_depth}  |  max_features: {model.max_features}')
print(f'  Fitur          : {FEATURE_COLS}')
print(f'  TFLite size    : {len(tflite_model)/1024:.1f} KB')
print(f'  {"─" * 52}')
print(f'  {"Metrik":<18} {"Sklearn":>10}  {"TFLite":>10}')
print(f'  {"─" * 42}')
print(f'  {"Accuracy":<18} {acc:>10.4f}  {tflite_acc:>10.4f}')
print(f'  {"F1-Score":<18} {f1:>10.4f}  {tflite_f1:>10.4f}')
print(f'  {"Precision":<18} {prec:>10.4f}  {tflite_prec:>10.4f}')
print(f'  {"Recall":<18} {rec:>10.4f}  {tflite_rec:>10.4f}')
print(f'{"=" * 62}')
print(f'\nSemua file tersimpan di: {SAVE_PATH}')
print(f'\nLangkah berikutnya:')
print(f'  1. Download url_classifier_rf_v2.tflite dari Drive')
print(f'  2. Salin ke: app/src/main/assets/url_classifier_rf.tflite')
print(f'  3. Build & install APK, lalu cek logcat tag "RFClassifier"')

## Cell 18 — Resume Pelatihan (Markdown Report)

In [ ]:
feat_imp_str = '\n'.join(
    f'  - {col}: {model.feature_importances_[i]:.4f}'
    for i, col in enumerate(FEATURE_COLS)
)

resume_lines = [
    '# Resume Pelatihan Random Forest URL Classifier',
    '',
    '## 1. Informasi Model',
    f'- **Jenis Model**: RandomForestClassifier (sklearn)',
    f'- **Jumlah Pohon**: {model.n_estimators}',
    f'- **Max Depth**: {model.max_depth}',
    f'- **Max Features**: {model.max_features}',
    f'- **Input**: 7 fitur leksikal URL',
    f'  - F1: domain_length, F2: digit_count, F3: dot_count, F4: delimiter_count',
    f'  - F5: suspicious_word_count (COUNT, bukan boolean)',
    f'  - F6: digit_letter_ratio, F7: max_sequential_digits',
    f'- **Output**: float32 [1, 2] — [P(aman), P(porno)]',
    '',
    '## 2. Dataset',
    f'- **Total URL**: {N_PER_CLASS * 2:,} (100K aman + 100K pornografi, balanced)',
    f'- **Pembagian**: 80% train / 20% test',
    f'- **Training samples**: {len(X_train):,}',
    f'- **Test samples**: {len(X_test):,}',
    '',
    '## 3. Feature Importance',
    feat_imp_str,
    '',
    '## 4. Hasil Evaluasi (Test Set)',
    '| Metrik | Sklearn | TFLite |',
    '|--------|---------|--------|',
    f'| Accuracy  | {acc:.4f} | {tflite_acc:.4f} |',
    f'| F1-Score  | {f1:.4f}  | {tflite_f1:.4f}  |',
    f'| Precision | {prec:.4f} | {tflite_prec:.4f} |',
    f'| Recall    | {rec:.4f}  | {tflite_rec:.4f}  |',
    f'| Delta Acc | —      | {abs(acc - tflite_acc):.4f}  |',
    '',
    '## 5. Model TFLite',
    f'- **Pipeline**: sklearn → ONNX → TFLite (Float16)',
    f'- **Ukuran**: {len(tflite_model) / 1024:.1f} KB',
    f'- **Input dtype**: FLOAT32 [1, 7]',
    f'- **Output dtype**: FLOAT32 [1, 2] — [P(aman), P(porno)]',
    f'- **Threshold**: 0.5',
    f'- **zipmap=False**: menghindari GATHER error di Android',
    '',
    '## 6. File Output',
    f'- `url_classifier_rf_v2.tflite` — model TFLite untuk Android',
    f'- `rf_model.onnx` — model ONNX (intermediate)',
    f'- `rf_model.joblib` — model sklearn (backup)',
    f'- `classification_report_rf.txt` / `.csv` — laporan per kelas',
    f'- `metrics_barchart_rf.png` — bar chart 4 metrik evaluasi',
    f'- `confusion_matrix_rf.png` — confusion matrix',
    f'- `feature_importance_rf.png` — visualisasi feature importance',
    f'- `model_info_rf.txt` — ringkasan informasi model',
    f'- `evaluation_metrics_rf.json` — semua metrik dalam format JSON',
    f'- `resume_rf.md` — laporan ini',
]

resume_md = '\n'.join(resume_lines)

resume_path = SAVE_PATH + 'resume_rf.md'
with open(resume_path, 'w', encoding='utf-8') as f:
    f.write(resume_md)

print(f'✅ Resume tersimpan: {resume_path}')
print()
print(resume_md)